# Reference-size reproduction on a GPU

The results of the article were obtained on a laptop CPU with coupling sub-networks of width 256 and depth 3.
This notebook reruns every experiment with the network size of the original BundleNet paper
(width 512, depth 5; Courts and Kvinge, ICLR 2022) on a Google Colab GPU.

Before running: select *Runtime > Change runtime type > GPU*, and place `creditcard.csv`
(see `data/README.md`) in your Google Drive at `My Drive/bundlenet/creditcard.csv`.

In [ ]:
!git clone https://github.com/Bra-AM/bundlenet_frauddetection.git
%cd bundlenet_frauddetection
# Colab's CUDA build of torch is kept; the other packages are pinned to the versions used in the article
!pip -q install FrEIA==0.2 geomloss==0.3.1 xgboost==3.4.1 lightgbm==4.7.0 imbalanced-learn==0.14.2 pot

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/My Drive/bundlenet/creditcard.csv" data/creditcard.csv
import torch
print('CUDA available:', torch.cuda.is_available())

## Configuration

`BN_EPOCHS=1000` takes roughly two to three hours per BundleNet seed on a T4 GPU; lower it for a quicker check.
Set `SEEDS = '0'` for a single-seed run.

In [ ]:
import os
os.environ.update(BN_WIDTH='512', BN_DEPTH='5', BN_EPOCHS='1000', BN_LR='2e-4', THREADS='2')
SEEDS = '0 1 2'
%cd src
!python data.py

## Training

In [ ]:
!python train_all.py score {SEEDS}
!MODELS=BundleNet,CGAN,CGAN-local python train_all.py class {SEEDS}

## Evaluation, figures and tables

In [ ]:
!python eval_generation.py score {SEEDS}
!python eval_generation.py class {SEEDS}
!SEEDS={SEEDS.replace(' ', ',')} python eval_augmentation.py
!SEEDS={SEEDS.replace(' ', ',')} python eval_classification.py
!python fiber_analysis.py
!python explanatory_figures.py
!python make_tables.py

In [ ]:
from IPython.display import Image, display
for name in ['fibers_illustration', 'high_risk_gallery', 'fiber_consistency', 'fiber_profiles', 'counterfactual_tradeoff']:
    display(Image(f'../results/figures/{name}.png'))

Copy the results to Google Drive so that they are kept after the session ends:

In [ ]:
!cp -r ../results "/content/drive/My Drive/bundlenet/results_reference_size"